In [ ]:
!pip install -q beir scikit-learn pandas numpy matplotlib

In [ ]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"

data_path = util.download_and_unzip(url, "data")

corpus, queries, qrels = GenericDataLoader(
    data_folder=data_path
).load(split="train")

print("Documents:", len(corpus))
print("Claims:", len(queries))
print("Relevance judgments:", len(qrels))

  0%|          | 0/5183 [00:00<?, ?it/s]

Documents: 5183
Claims: 809
Relevance judgments: 809


In [ ]:
# Ensure data is loaded and packages are installed if previous cells weren't run in this session
if 'corpus' not in globals() or 'queries' not in globals() or 'qrels' not in globals():
    print("Variables not found in memory. Checking dependencies...")
    try:
        from beir import util
        from beir.datasets.data_loader import GenericDataLoader
    except ModuleNotFoundError:
        print("BEIR not installed. Installing beir package now...")
        import sys
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "beir"])
        from beir import util
        from beir.datasets.data_loader import GenericDataLoader

    url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"
    data_path = util.download_and_unzip(url, "data")
    corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="train")

# Inspect one scientific abstract
doc_id, doc = list(corpus.items())[0]

print("Document ID:", doc_id)
print("Title:", doc["title"])
print("Abstract:", doc["text"])

print("\n" + "="*60 + "\n")

# Inspect one claim
claim_id, claim = list(queries.items())[0]

print("Claim ID:", claim_id)
print("Claim:", claim)

print("\n" + "="*60 + "\n")

# Inspect relevance judgments (qrels is a dict mapping query_id to dict of doc_id -> score)
print("Relevance judgments (sample keys and values):")
print(list(qrels.items())[:5])

Document ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Abstract: Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the m

In [ ]:
query_id = str(claim_id)

relevant_docs = qrels[query_id]

print("Claim:", queries[query_id])

for doc_id, score in relevant_docs.items():
    print("\nRelevance score:", score)
    print("Document ID:", doc_id)
    print("Title:", corpus[doc_id]["title"])
    print("Abstract:", corpus[doc_id]["text"])

Claim: 0-dimensional biomaterials lack inductive properties.

Relevance score: 1
Document ID: 31715818
Title: New opportunities: the use of nanotechnologies to manipulate and track stem cells.
Abstract: Nanotechnologies are emerging platforms that could be useful in measuring, understanding, and manipulating stem cells. Examples include magnetic nanoparticles and quantum dots for stem cell labeling and in vivo tracking; nanoparticles, carbon nanotubes, and polyplexes for the intracellular delivery of genes/oligonucleotides and protein/peptides; and engineered nanometer-scale scaffolds for stem cell differentiation and transplantation. This review examines the use of nanotechnologies for stem cell tracking, differentiation, and transplantation. We further discuss their utility and the potential concerns regarding their cytotoxicity.


In [ ]:
doc_ids = list(corpus.keys())

documents = []

for doc_id in doc_ids:
    title = corpus[doc_id]["title"]
    abstract = corpus[doc_id]["text"]

    documents.append(title + " " + abstract)

print("Total documents:", len(documents))
print("\nFirst document:\n", documents[0][:500])

Total documents: 5183

First document:
 Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to deline


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(documents)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (5183, 50000)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
def retrieve_documents(claim, top_k=5):
    # Step 1: Convert claim into TF-IDF vector
    query_vector = vectorizer.transform([claim])

    # Step 2: Calculate cosine similarity with all documents
    # cosine_similarity returns a 2D array of shape (1, num_documents)
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Step 3: Get indices of top_k highest scores
    # np.argsort sorts in ascending order, so we reverse it with [::-1]
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Step 4: Prepare results
    results = []

    for idx in top_indices:
        results.append({
            "doc_id": doc_ids[idx],
            "score": float(scores[idx]),
            "title": corpus[doc_ids[idx]]["title"]
        })

    return results

In [ ]:
claim = "Vitamin D supplementation reduces respiratory infections"

results = retrieve_documents(claim, top_k=5)

for result in results:
    print(result)

{'doc_id': '23267371', 'score': 0.18289575884817338, 'title': 'Vitamin D: The "sunshine" vitamin.'}
{'doc_id': '21553394', 'score': 0.1576310233740233, 'title': 'Vitamin D and obesity: current perspectives and future directions.'}
{'doc_id': '12009265', 'score': 0.1346817466583381, 'title': "Vitamins E and C in the prevention of prostate and total cancer in men: the Physicians' Health Study II randomized controlled trial."}
{'doc_id': '9167230', 'score': 0.11251516764867665, 'title': 'Global and regional burden of hospital admissions for severe acute lower respiratory infections in young children in 2010: a systematic analysis'}
{'doc_id': '22843838', 'score': 0.10535290401340561, 'title': '[Vitamin D and latitude as environmental factors in multiple sclerosis].'}


In [ ]:
query_id = list(queries.keys())[0]

relevant_docs = set(qrels[query_id].keys())

print("Query:", queries[query_id])
print("Relevant document IDs:", relevant_docs)
print("Total relevant documents:", len(relevant_docs))

Query: 0-dimensional biomaterials lack inductive properties.
Relevant document IDs: {'31715818'}
Total relevant documents: 1


In [ ]:
def calculate_recall(query_id, top_k=5):
    claim = queries[query_id]
    results = retrieve_documents(claim, top_k)
    retrieved_ids = set(result["doc_id"] for result in results)
    relevant_ids = set(qrels[query_id].keys())

    # Calculate intersection of retrieved and relevant IDs
    intersection = retrieved_ids.intersection(relevant_ids)

    # Calculate recall (avoid division by zero if relevant_ids is empty)
    if len(relevant_ids) == 0:
        recall = 0.0
    else:
        recall = len(intersection) / len(relevant_ids)

    return recall

In [ ]:
# Test the calculate_recall function on the first query
first_query_id = list(queries.keys())[0]
recall_score = calculate_recall(first_query_id, top_k=5)
print(f"Recall@{5} for query '{first_query_id}': {recall_score:.2f}")

Recall@5 for query '0': 0.00


In [ ]:
query_id = "0"

relevant_ids = set(qrels[query_id].keys())

results = retrieve_documents(queries[query_id], top_k=5)

retrieved_ids = set(
    str(result["doc_id"]) for result in results
)

print("Relevant IDs:", relevant_ids)
print("Retrieved IDs:", retrieved_ids)

print("Intersection:", retrieved_ids.intersection(relevant_ids))

Relevant IDs: {'31715818'}
Retrieved IDs: {'825728', '10607877', '27049238', '40212412', '28138927'}
Intersection: set()


In [ ]:
all_results = retrieve_documents(queries[query_id], top_k=len(corpus))

for rank, result in enumerate(all_results, start=1):
    if str(result["doc_id"]) in relevant_ids:
        print("Relevant document rank:", rank)
        print("Title:", result["title"])
        print("Score:", result["score"])
        break

Relevant document rank: 2830
Title: New opportunities: the use of nanotechnologies to manipulate and track stem cells.
Score: 0.0


In [ ]:
recalls = []

for query_id in queries:

    recall = calculate_recall(query_id, top_k=5)

    recalls.append(recall)

mean_recall = np.mean(recalls)

print("Mean Recall@5:", mean_recall)

Mean Recall@5: 0.6781417387721467


In [ ]:
baseline_results = {
    "Model": "TF-IDF",
    "Mean Recall@5": mean_recall
}

print(baseline_results)

{'Model': 'TF-IDF', 'Mean Recall@5': np.float64(0.6781417387721467)}


In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
sample = "Vitamin D supplementation reduces respiratory infections"

embedding = model.encode(sample)

print("Embedding shape:", embedding.shape)
print("First 10 values:", embedding[:10])

Embedding shape: (384,)
First 10 values: [-0.03531391  0.04992547  0.01969674 -0.03140105  0.03327258 -0.01393718
  0.01147121  0.08239119 -0.00519954  0.05140042]


In [ ]:
document_embeddings = model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding matrix shape:", document_embeddings.shape)

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Embedding matrix shape: (5183, 384)


In [ ]:
def semantic_retrieve(claim, top_k=5):
    # 1. Generate embedding for the input claim
    query_embedding = model.encode(claim, normalize_embeddings=True)

    # 2. Calculate similarity with all document embeddings
    # Since both embeddings are normalized, the dot product is equivalent to cosine similarity
    scores = np.dot(document_embeddings, query_embedding)

    # 3. Get indices of top_k highest scores
    top_indices = np.argsort(scores)[::-1][:top_k]

    # 4. Store results
    results = []

    for idx in top_indices:
        results.append({
            "doc_id": doc_ids[idx],
            "score": float(scores[idx]),
            "title": corpus[doc_ids[idx]]["title"]
        })

    return results

In [ ]:
# Test semantic_retrieve on the same sample claim
semantic_results = semantic_retrieve(claim, top_k=5)
for result in semantic_results:
    print(result)

{'doc_id': '23267371', 'score': 0.5674275159835815, 'title': 'Vitamin D: The "sunshine" vitamin.'}
{'doc_id': '36960449', 'score': 0.5306119918823242, 'title': 'Estimation of the dietary requirement for vitamin D in healthy adults.'}
{'doc_id': '30720103', 'score': 0.51678466796875, 'title': 'Vitamin D status: measurement, interpretation, and clinical application.'}
{'doc_id': '22843838', 'score': 0.5005931854248047, 'title': '[Vitamin D and latitude as environmental factors in multiple sclerosis].'}
{'doc_id': '8856690', 'score': 0.4717804789543152, 'title': 'Molecular Mechanisms of Vitamin D Action'}


In [ ]:
# 1. Redefine a recall function specifically for semantic retrieval
def calculate_semantic_recall(query_id, top_k=5):
    claim = queries[query_id]
    results = semantic_retrieve(claim, top_k)
    retrieved_ids = set(result["doc_id"] for result in results)
    relevant_ids = set(qrels[query_id].keys())

    intersection = retrieved_ids.intersection(relevant_ids)
    if len(relevant_ids) == 0:
        return 0.0
    return len(intersection) / len(relevant_ids)

# 2. Evaluate Mean Recall@5 for all queries
semantic_recalls = []
for query_id in queries:
    recall = calculate_semantic_recall(query_id, top_k=5)
    semantic_recalls.append(recall)

mean_semantic_recall = np.mean(semantic_recalls)

# 3. Print and compare results
results_comparison = {
    "TF-IDF Recall@5": mean_recall,
    "Semantic Recall@5": mean_semantic_recall
}
print("Evaluation Comparison:", results_comparison)

Evaluation Comparison: {'TF-IDF Recall@5': np.float64(0.6781417387721467), 'Semantic Recall@5': np.float64(0.7347342398022251)}


In [ ]:
# 1. Redefine a recall function specifically for semantic retrieval
def calculate_semantic_recall(query_id, top_k=5):
    claim = queries[query_id]
    results = semantic_retrieve(claim, top_k)
    retrieved_ids = set(result["doc_id"] for result in results)
    relevant_ids = set(qrels[query_id].keys())

    intersection = retrieved_ids.intersection(relevant_ids)
    if len(relevant_ids) == 0:
        return 0.0
    return len(intersection) / len(relevant_ids)

# 2. Evaluate Mean Recall@5 for all queries
semantic_recalls = []
for query_id in queries:
    recall = calculate_semantic_recall(query_id, top_k=5)
    semantic_recalls.append(recall)

mean_semantic_recall = np.mean(semantic_recalls)

# 3. Print and compare results
results_comparison = {
    "TF-IDF Recall@5": mean_recall,
    "Semantic Recall@5": mean_semantic_recall
}
print("Evaluation Comparison:", results_comparison)

Evaluation Comparison: {'TF-IDF Recall@5': np.float64(0.6781417387721467), 'Semantic Recall@5': np.float64(0.7347342398022251)}


In [ ]:
claim = queries["0"]

results = semantic_retrieve(claim, top_k=5)

for result in results:
    print(result)

{'doc_id': '29638116', 'score': 0.3453496992588043, 'title': 'Complex Tissue and Disease Modeling using hiPSCs.'}
{'doc_id': '4346436', 'score': 0.333349347114563, 'title': 'Nonlinear Elasticity in Biological Gels'}
{'doc_id': '17388232', 'score': 0.3129124045372009, 'title': 'Mechanical regulation of cell function with geometrically modulated elastomeric substrates'}
{'doc_id': '31715818', 'score': 0.28793781995773315, 'title': 'New opportunities: the use of nanotechnologies to manipulate and track stem cells.'}
{'doc_id': '927561', 'score': 0.2870509624481201, 'title': 'Emergent structures and dynamics of cell colonies by contact inhibition of locomotion'}


In [ ]:
def calculate_semantic_recall(query_id, top_k=5):

    claim = queries[query_id]

    results = semantic_retrieve(claim, top_k)

    retrieved_ids = set(
        str(result["doc_id"]) for result in results
    )

    relevant_ids = set(
        str(doc_id) for doc_id in qrels[query_id].keys()
    )

    if len(relevant_ids) == 0:
        return 0

    recall = len(
        retrieved_ids.intersection(relevant_ids)
    ) / len(relevant_ids)

    return recall

In [ ]:
semantic_recalls = []

for query_id in queries:

    recall = calculate_semantic_recall(
        query_id,
        top_k=5
    )

    semantic_recalls.append(recall)

semantic_mean_recall = np.mean(semantic_recalls)

print("TF-IDF Recall@5:", mean_recall)
print("Sentence-BERT Recall@5:", semantic_mean_recall)

TF-IDF Recall@5: 0.6781417387721467
Sentence-BERT Recall@5: 0.7347342398022251


In [ ]:
improvement = (
    semantic_mean_recall - mean_recall
) * 100

print("Improvement:", round(improvement, 2), "percentage points")

Improvement: 5.66 percentage points


In [ ]:
# Using the globally defined 'embedding' variable instead of the function-local 'query_embedding'
scores = np.dot(document_embeddings, embedding)

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss

dimension = document_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(document_embeddings.astype("float32"))

print("Total indexed documents:", index.ntotal)

Total indexed documents: 5183


In [ ]:
query_embedding = model.encode(
    queries["0"],
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(query_embedding.reshape(1, -1), 5)

for score, idx in zip(scores[0], indices[0]):
    print(
        doc_ids[idx],
        score,
        corpus[doc_ids[idx]]["title"]
    )

29638116 0.34534967 Complex Tissue and Disease Modeling using hiPSCs.
4346436 0.33334935 Nonlinear Elasticity in Biological Gels
17388232 0.31291243 Mechanical regulation of cell function with geometrically modulated elastomeric substrates
31715818 0.28793782 New opportunities: the use of nanotechnologies to manipulate and track stem cells.
927561 0.28705096 Emergent structures and dynamics of cell colonies by contact inhibition of locomotion
